<center> <h1>GraphRAG with Unstructured.io and Neo4j</h1>

## 1. Requirements

In [ ]:
!pip install unstructured-client
!pip install neo4j
!pip install neo4j-graphrag
!pip install python-dotenv

## 2. Get your free API key

- Get your Unstructured Serverless API key for a free 14 day trial period with **1000 pages/day** here: https://unstructured.io/api-key-hosted  

- Get a free API key with **1000 pages/month** here: https://unstructured.io/api-key-free
- Official API documentation : https://docs.unstructured.io/api-reference/api-services/overview
- Official chunking documentation : https://docs.unstructured.io/api-reference/api-services/chunking#basic-chunking-strategy

## 2. Get your free API key

## 2. Get your free API key

In [11]:
import os
from dotenv import load_dotenv

# .env variables
load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
unstructured_api_key= os.getenv("UNSTRUCTURED_API_KEY")
neo4j_uri = os.getenv("NEO4J_URI")
neo4j_database = os.getenv("NEO4J_DATABASE")
neo4j_user = os.getenv("NEO4J_USERNAME")
neo4j_password = os.getenv("NEO4J_PASSWORD")

## 2. Get your free API key

In [12]:
from neo4j import GraphDatabase

# Neo4j driver setup
driver = GraphDatabase.driver(neo4j_uri, auth=(neo4j_user, neo4j_password))

# Cypher for index creation
VECTOR_INDEX_QUERY = '''
CREATE VECTOR INDEX chunk_embedding IF NOT EXISTS
FOR (n:Chunk) ON (n.embedding)
OPTIONS {
  indexConfig: {
    `vector.dimensions`: 1536,
    `vector.similarity_function`: 'cosine'
  }
}
'''

FULLTEXT_INDEX_QUERY = '''
CREATE FULLTEXT INDEX entity_text IF NOT EXISTS
FOR (n:Entity) ON EACH [n.text, n.variants]
OPTIONS {
  indexConfig: {
    `fulltext.analyzer`: 'english',
    `fulltext.eventually_consistent`: true
  }
}
'''

def create_index(query: str, label: str):
    try:
        with driver.session() as session:
            result = session.run(query)
            summary = result.consume()

            print(f"\n Index for {label} created (or already exists).")
            print("Query:\n", summary.query.text)
            print("Indexes added:", summary.counters.indexes_added)
            if summary.notifications:
                print("Notifications:", summary.notifications)

    except Exception as e:
        print(f" Failed to create index for {label}: {e}")

def main():
    create_index(VECTOR_INDEX_QUERY, "Chunk.embedding vector")
    create_index(FULLTEXT_INDEX_QUERY, "Entity fulltext")

    driver.close()
    print("\nDone creating indexes.")

if __name__ == "__main__":
    main()



 Index for Chunk.embedding vector created (or already exists).
 Failed to create index for Chunk.embedding vector: 'str' object has no attribute 'text'

 Index for Entity fulltext created (or already exists).
 Failed to create index for Entity fulltext: 'str' object has no attribute 'text'

Done creating indexes.


## 3. Chunk PDFs with Unstructured into Neo4j

In [252]:
import os
import base64
import zlib
import json
import logging
import nltk

from neo4j import GraphDatabase
from unstructured_client import UnstructuredClient
from unstructured_client.models import operations, shared
from unstructured.staging.base import elements_from_dicts, elements_to_json

# Disable logging
logging.disable(logging.CRITICAL)

# Configuration
directory_path = "/Users/michaelmoore/Projects/oxy/15-9-F14/input/"

client = UnstructuredClient(
    api_key_auth=unstructured_api_key,
    server_url="https://api.unstructuredapp.io"
)

driver = GraphDatabase.driver(neo4j_uri, auth=(neo4j_user, neo4j_password))

CHUNK_QUERY = '''
WITH apoc.convert.fromJsonList($json) AS maps
UNWIND maps AS map
WITH apoc.map.clean(map,[],["  ",""]) AS m
MERGE (d:Document {name: m.metadata.filename})
WITH m, d
CALL(m, d) {
  CREATE (n:Chunk {id: m.element_id})
  SET
    n.type = "NarrativeText",
    n.text = m.text,
    n.filename = m.metadata.filename,
    n.filetype = m.metadata.filetype,
    n.languages = m.metadata.languages,
    n.page_number = m.metadata.page_number,
    n.tokens = m.tokens
  CREATE (n)-[:PART_OF_DOCUMENT]->(d)
  RETURN n
}
WITH m, d, n
CALL(m, d, n) {
  WITH m, d, n
  WHERE m.metadata.type IN ['Image', 'Table']
  CREATE (i:$(m.metadata.type) {id: m.element_id})
  SET i.type = m.metadata.type,
      i.figure_caption = m.metadata.figure_caption,
      i.text = m.metadata.text,
      i.filename = m.metadata.filename,
      i.filetype = m.metadata.filetype,
      i.languages = m.metadata.languages,
      i.page_number = m.metadata.page_number,
      i.image_base64 = m.metadata.image_base64,
      i.image_mime_type = m.metadata.image_mime_type,
      i.text_as_html = m.metadata.text_as_html
  MERGE (n)-[:RELATED_CONTENT]->(i)
  MERGE (i)-[:PART_OF_DOCUMENT]->(d)
}
WITH DISTINCT d, n
WITH d, COLLECT(n) AS nodes
CALL apoc.nodes.link(nodes, "NEXT_CHUNK")
'''

def run_query(tx, query, json_data):
    return tx.run(query, {"json": json_data}).consume()

def extract_orig_elements(encoded):
    decoded = base64.b64decode(encoded)
    decompressed = zlib.decompress(decoded)
    return json.loads(decompressed.decode("utf-8"))

def process_file(filepath, filename):
    print(f"\nProcessing file: {filename}")
    
    with open(filepath, "rb") as f:
        files = shared.Files(
            content=f.read(),
            file_name=filename
        )

    request = operations.PartitionRequest(
        partition_parameters=shared.PartitionParameters(
            files=files,
            strategy="hi_res",
            hi_res_model_name="yolox",
            element_exclude=['Header', 'Footer', 'ListItem', 'Formula', 'UncategorizedText'],
            extract_image_block_types=['Image', 'Table'],
            chunking_strategy='by_title',
            max_characters=1500,
            split_pdf_page=True,
            split_pdf_allow_failed=True,
            split_pdf_concurrency_level=15
        )
    )

    response = client.general.partition(request=request)
    element_dicts = [e for e in response.elements]

    for i, element in enumerate(element_dicts):
        if element.get("text"):
            element["tokens"] = len(nltk.word_tokenize(element["text"]))

        metadata = element.get("metadata", {})
        if metadata.get("orig_elements"):
            orig_elements = extract_orig_elements(metadata["orig_elements"])

            for obj in orig_elements:
                if obj.get("type") == "FigureCaption" and obj.get("text", "").lower().startswith("figure"):
                    metadata["figure_caption"] = obj["text"]

                if obj.get("type") == "Image":
                    metadata.update({
                        "element_id": obj["element_id"],
                        "type": obj["type"],
                        "image_base64": obj["metadata"]["image_base64"],
                        "image_mime_type": obj["metadata"]["image_mime_type"],
                        "text": obj["text"]
                    })

                if obj.get("type") == "Table":
                    metadata.update({
                        "element_id": obj["element_id"],
                        "type": obj["type"],
                        "text_as_html": obj["metadata"]["text_as_html"],
                        "image_base64": obj["metadata"]["image_base64"],
                        "image_mime_type": obj["metadata"]["image_mime_type"],
                        "text": obj["text"]
                    })

        element_dicts[i]['metadata'].pop('orig_elements', None)

    json_data = json.dumps(element_dicts, indent=4)

    with driver.session() as session:
        summary = session.execute_write(run_query, CHUNK_QUERY, json_data)
        print(f"nodes created => {summary.counters.nodes_created}, rels created => {summary.counters.relationships_created}")
    session.close() 
    print(f"Finished processing: {filename}")

def main():
    for filename in os.listdir(directory_path):
        if filename.startswith('.') or not os.path.isfile(os.path.join(directory_path, filename)):
            continue

        try:
            process_file(os.path.join(directory_path, filename), filename)
        except Exception as e:
            print(f"Error processing {filename}: {e}")

    print("Done!")

if __name__ == "__main__":
    main()


Processing file: Statoil_Volve_15_9F14_Preliminaryjob_Presentation.pdf
nodes created => 27, rels created => 39
Finished processing: Statoil_Volve_15_9F14_Preliminaryjob_Presentation.pdf
Done!


## 4. Generate Vector Embeddings for Chunks

In [256]:
from neo4j import GraphDatabase
from neo4j_graphrag.embeddings import OpenAIEmbeddings
import logging
import sys

# Disable logging output
logging.disable(sys.maxsize)

# --- Configuration ---
EMBEDDING_MODEL = "text-embedding-ada-002"
MAX_CHUNK_LENGTH = 12000

# Initialize OpenAI embedder
embedder = OpenAIEmbeddings(model=EMBEDDING_MODEL, api_key=openai_api_key)

# Initialize Neo4j driver
driver = GraphDatabase.driver(neo4j_uri, auth=(neo4j_user, neo4j_password))


def embed_chunks():
    with driver.session() as session:
        result = session.run("""
            MATCH (n:Chunk)
            WHERE n.text IS NOT NULL AND n.embedding IS NULL
            RETURN n.id AS id, n.text AS data
        """)

        for record in result:
            chunk_id = record["id"]
            data = record["data"]

            if len(data) > MAX_CHUNK_LENGTH:
                print(f"Skipping chunk {chunk_id} (length: {len(data)})")
                continue

            print(f"\rEmbedding chunk: {chunk_id}", end="", flush=True)

            try:
                embedding = embedder.embed_query(data)

                session.run("""
                    MATCH (n:Chunk {id: $chunk_id})
                    SET n.embedding = $embedding
                """, chunk_id=chunk_id, embedding=embedding)

            except Exception as e:
                print(f"\nFailed to embed chunk {chunk_id}: {e}")


def main():
    embed_chunks()
    driver.close()
    print("\nDone embedding chunks!")


if __name__ == "__main__":
    main()     


Embedding chunk: 3702e0da542792ba05ceac6985cecf90
Done embedding chunks!


## 5. Entity Extraction for Full Text Search

In [257]:
#For this demo we'll perform entity extract on only the :NarrativeText chunks
#Entity Extraction Prep (run once)

from neo4j import GraphDatabase

# Neo4j driver setup
driver = GraphDatabase.driver(neo4j_uri, auth=(neo4j_user, neo4j_password))

# to perform selective entity extraction using "ProcessMe" label execute this query

PROCESS_ME ='''
MATCH (n:Chunk {type:"NarrativeText"}) 
WHERE NOT (n)-[:HAS_ENTITY]->() AND n.entities IS NULL
SET n:ProcessMe
'''

with driver.session() as session:
    res = session.run(PROCESS_ME)
session.close() 
print("done!")

done!


In [258]:
import logging
import sys
from neo4j import GraphDatabase
from openai import OpenAI

# --- Config ---
MAX_CHUNKS = 1000
OPENAI_MODEL = "gpt-4o"  # or "gpt-4"

# --- Logging ---
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logging.disable(sys.maxsize)  # Disable logging if needed

# --- Clients ---
client = OpenAI(api_key=openai_api_key)
driver = GraphDatabase.driver(neo4j_uri, auth=(neo4j_user, neo4j_password))

# --- Cypher Queries ---
FETCH_CHUNKS_QUERY = """
MATCH (n:Chunk:ProcessMe)
WHERE NOT (n)-[:HAS_ENTITY]->() AND n.entities IS NULL
RETURN n.id AS id, replace(n.text,"\n","") AS text
LIMIT $limit
"""

ENTITY_INSERT_QUERY = """
WITH $entities AS entities
MATCH (n:Chunk:ProcessMe {id: $id})
WITH n, entities
CALL apoc.do.when(
    entities[0] = "[]" OR entities[0] STARTS WITH "The text provided does not contain",
    "WITH n SET n.entities = 'failed' REMOVE n:ProcessMe RETURN 0 AS rels",
    "WITH n, apoc.convert.fromJsonList(entities[0]) AS names
     UNWIND names AS name
     MERGE (e:Entity {text: toLower(name)})
     ON CREATE SET e.variants = [name]
     ON MATCH SET e.variants = apoc.convert.toSet(e.variants + [name])
     MERGE (n)-[:HAS_ENTITY]->(e)
     WITH DISTINCT n, COUNT(e) AS rels
     REMOVE n:ProcessMe
     RETURN rels",
    {n: n, entities: entities}
) YIELD value
RETURN value
"""

FAIL_MARK_QUERY = """
MATCH (n:Chunk:ProcessMe {id: $id})
SET n.entities = "failed"
REMOVE n:ProcessMe
"""

# --- Entity Extraction Prompt ---
def extract_entities(text: str) -> list:
    prompt = f"""
Extract all the entities from the following text. Identify only entities, abbreviations and technical terms commonly used in the petroleum exploration, petroleum geology, reservoir analysis, and oil & gas production.

Return entities in this format: ["entity1", "entity2"]

Do not include any extra text or explanation.

Text: {text}
"""

    messages = [
        {"role": "system", "content": "You help extract entities from petroleum-related text."},
        {"role": "user", "content": prompt}
    ]

    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=messages,
        max_tokens=500,
        temperature=0
    )

    return [response.choices[0].message.content.strip()]


# --- Main Function ---
def main():
    processed_count = 0

    with driver.session() as session:
        chunks = session.run(FETCH_CHUNKS_QUERY, limit=MAX_CHUNKS)

        for chunk in chunks:
            chunk_id = chunk["id"]
            text = chunk["text"]
            processed_count += 1

            try:
                entities = extract_entities(text)
                result = session.run(ENTITY_INSERT_QUERY, id=chunk_id, entities=entities)

                for record in result:
                    rels = record["value"]["rels"]
                    print(f"\rRelationships created: {rels} | Processed: {processed_count}     ", end="", flush=True)

                result.consume()

            except Exception as e:
                logging.warning(f"\nFailed to process chunk {chunk_id}: {e}")
                session.run(FAIL_MARK_QUERY, id=chunk_id)

    print("\nDone extracting entities!")


if __name__ == "__main__":
    main()


Relationships created: 13 | Processed: 13     
Done extracting entities!


## 6. Some Cleanup

In [261]:
import base64
from io import BytesIO
from PIL import Image
from neo4j import GraphDatabase
import json

driver = GraphDatabase.driver(neo4j_uri, auth=(neo4j_user, neo4j_password))

# Cypher query to fetch nodes with base64 images and no byte size info yet
QUERY_IMAGES= """
MATCH (n:Image|Table)
WHERE n.image_base64 IS NOT NULL AND n.bytes IS NULL
RETURN n.id AS id, n.image_base64 AS image_base64
"""

def get_image_properties(image_base64: str):
    try:
        image_data = base64.b64decode(image_base64)
        with Image.open(BytesIO(image_data)) as image:
            width, height = image.size
            aspect_ratio = max(width / height, height / width) if width and height else None
            return {
                "bytes": len(image_base64),
                "width": width,
                "height": height,
                "aspect_ratio": aspect_ratio
            }
    except Exception as e:
        print(f"Error processing image: {e}")
        return None

def update_image_properties(driver):
    with driver.session() as session:
        result = session.run(QUERY_IMAGES)
        
        for record in result:
            node_id = record["id"]
            image_base64 = record["image_base64"]
            props = get_image_properties(image_base64)

            if props:
                session.run(
                    """
                    WITH apoc.convert.fromJsonMap($json) AS map
                    MATCH (n:Image|Table {id: $id})
                    SET n += map
                    """,
                    {"id": node_id, "json": json.dumps(props)}
                )

if __name__ == "__main__":
    update_image_properties(driver)
    print("Image metadata updated.")

Image metadata updated.
